<a href="https://colab.research.google.com/github/fathimathulaifa/ICT-assessments-and-assignments/blob/main/Intermediate_Assessment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

predict loan approval status

In [1]:
import pandas as pd
import numpy as np

In [10]:
# Load Data
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")


In [11]:
train.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [12]:
test.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,LP001015,Male,Yes,0,Graduate,No,5720,0,110.0,360.0,1.0,Urban
1,LP001022,Male,Yes,1,Graduate,No,3076,1500,126.0,360.0,1.0,Urban
2,LP001031,Male,Yes,2,Graduate,No,5000,1800,208.0,360.0,1.0,Urban
3,LP001035,Male,Yes,2,Graduate,No,2340,2546,100.0,360.0,NaN,Urban
4,LP001051,Male,No,0,Not Graduate,No,3276,0,78.0,360.0,1.0,Urban


In [6]:
train.shape



(367, 12)

In [7]:
test.shape

(614, 13)

In [13]:
train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [19]:
train = train.drop("Loan_ID", axis=1) # dropping loan_id as it is not needed when training

In [20]:
train.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [16]:
test_ids = test["Loan_ID"] # saving it loan_id to predict it later to test


In [17]:
test = test.drop("Loan_ID", axis=1)

In [23]:
# Categorical columns
categorical_cols = train.select_dtypes(include='object').columns
print("Categorical Columns:")
print(categorical_cols)

Categorical Columns:
Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area', 'Loan_Status'],
      dtype='object')


In [24]:
#loan_status is a target so we remove it from categorical_cols
categorical_cols = categorical_cols.drop("Loan_Status")

In [25]:
print(categorical_cols)

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area'],
      dtype='object')


In [26]:
# Numerical Columns
numerical_cols = train.select_dtypes(include=['int64','float64']).columns
print(numerical_cols)


Index(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History'],
      dtype='object')


In [28]:
# Gender
train["Gender"] = train["Gender"].fillna(train["Gender"].mode()[0])
test["Gender"] = test["Gender"].fillna(train["Gender"].mode()[0])

In [29]:
# Married
train["Married"] = train["Married"].fillna(train["Married"].mode()[0])
test["Married"] = test["Married"].fillna(train["Married"].mode()[0])

In [30]:
# Dependents
train["Dependents"] = train["Dependents"].fillna(train["Dependents"].mode()[0])
test["Dependents"] = test["Dependents"].fillna(train["Dependents"].mode()[0])

In [32]:
# Self_Employed
train["Self_Employed"] = train["Self_Employed"].fillna(train["Self_Employed"].mode()[0])
test["Self_Employed"] = test["Self_Employed"].fillna(train["Self_Employed"].mode()[0])

In [36]:
print(numerical_cols)

Index(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History'],
      dtype='object')


In [33]:
# Fill Missing Numerical Values-> median
train["LoanAmount"] = train["LoanAmount"].fillna(train["LoanAmount"].median())
test["LoanAmount"] = test["LoanAmount"].fillna(train["LoanAmount"].median())

In [34]:
train["Loan_Amount_Term"] = train["Loan_Amount_Term"].fillna(train["Loan_Amount_Term"].median())
test["Loan_Amount_Term"] = test["Loan_Amount_Term"].fillna(train["Loan_Amount_Term"].median())

In [35]:
train["Credit_History"] = train["Credit_History"].fillna(train["Credit_History"].mode()[0])
test["Credit_History"] = test["Credit_History"].fillna(train["Credit_History"].mode()[0])

In [37]:
train.isnull().sum()


,0
Gender,0
Married,0
Dependents,0
Education,0
Self_Employed,0
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,0
Loan_Amount_Term,0
Credit_History,0


In [38]:
test.isnull().sum()


,0
Gender,0
Married,0
Dependents,0
Education,0
Self_Employed,0
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,0
Loan_Amount_Term,0
Credit_History,0


In [39]:
print(train["Gender"].unique())
print(train["Married"].unique())
print(train["Education"].unique())
print(train["Self_Employed"].unique())
print(train["Loan_Status"].unique())


['Male' 'Female']
['No' 'Yes']
['Graduate' 'Not Graduate']
['No' 'Yes']
['Y' 'N']


In [40]:
# Encoding -> labelencoding
train["Gender"] = train["Gender"].map({"Male": 1, "Female": 0})
test["Gender"] = test["Gender"].map({"Male": 1, "Female": 0})

In [41]:
train["Married"] = train["Married"].map({"Yes": 1, "No": 0})
test["Married"] = test["Married"].map({"Yes": 1, "No": 0})

In [42]:
train["Education"] = train["Education"].map({"Graduate": 1, "Not Graduate": 0})
test["Education"] = test["Education"].map({"Graduate": 1, "Not Graduate": 0})

In [43]:
train["Self_Employed"] = train["Self_Employed"].map({"Yes": 1, "No": 0})
test["Self_Employed"] = test["Self_Employed"].map({"Yes": 1, "No": 0})

In [44]:
train["Loan_Status"] = train["Loan_Status"].map({"Y": 1, "N": 0})

In [45]:
train.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,1,0,0,1,0,5849,0.0,128.0,360.0,1.0,Urban,1
1,1,1,1,1,0,4583,1508.0,128.0,360.0,1.0,Rural,0
2,1,1,0,1,1,3000,0.0,66.0,360.0,1.0,Urban,1
3,1,1,0,0,0,2583,2358.0,120.0,360.0,1.0,Urban,1
4,1,0,0,1,0,6000,0.0,141.0,360.0,1.0,Urban,1


In [46]:
train["Dependents"].unique()


array(['0', '1', '2', '3+'], dtype=object)

In [47]:
train["Property_Area"].unique()


array(['Urban', 'Rural', 'Semiurban'], dtype=object)

In [48]:
# encoding->one-hot
train = pd.get_dummies(train, columns=["Dependents", "Property_Area"], drop_first=True)
test = pd.get_dummies(test, columns=["Dependents", "Property_Area"], drop_first=True)


In [54]:
# align columns
train, test = train.align(test, join="left", axis=1, fill_value=0)


In [56]:
test = test.drop("Loan_Status", axis=1)


In [59]:
print(train.shape)
print(test.shape)


(614, 15)
(367, 14)


In [60]:
print("Columns in train:")
print(train.columns)

print("Columns in test:")
print(test.columns)


Columns in train:
Index(['Gender', 'Married', 'Education', 'Self_Employed', 'ApplicantIncome',
       'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History',
       'Loan_Status', 'Dependents_1', 'Dependents_2', 'Dependents_3+',
       'Property_Area_Semiurban', 'Property_Area_Urban'],
      dtype='object')
Columns in test:
Index(['Gender', 'Married', 'Education', 'Self_Employed', 'ApplicantIncome',
       'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History',
       'Dependents_1', 'Dependents_2', 'Dependents_3+',
       'Property_Area_Semiurban', 'Property_Area_Urban'],
      dtype='object')


Separate Features and Target

In [63]:
X = train.drop("Loan_Status", axis=1)
y = train["Loan_Status"] # target


In [64]:
X.head()



,Gender,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Dependents_1,Dependents_2,Dependents_3+,Property_Area_Semiurban,Property_Area_Urban
0,1,0,1,0,5849,0.0,128.0,360.0,1.0,False,False,False,False,True
1,1,1,1,0,4583,1508.0,128.0,360.0,1.0,True,False,False,False,False
2,1,1,1,1,3000,0.0,66.0,360.0,1.0,False,False,False,False,True
3,1,1,0,0,2583,2358.0,120.0,360.0,1.0,False,False,False,False,True
4,1,0,1,0,6000,0.0,141.0,360.0,1.0,False,False,False,False,True


In [65]:
X = X.astype(int)
test = test.astype(int)


In [67]:
X.head()

,Gender,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Dependents_1,Dependents_2,Dependents_3+,Property_Area_Semiurban,Property_Area_Urban
0,1,0,1,0,5849,0,128,360,1,0,0,0,0,1
1,1,1,1,0,4583,1508,128,360,1,1,0,0,0,0
2,1,1,1,1,3000,0,66,360,1,0,0,0,0,1
3,1,1,0,0,2583,2358,120,360,1,0,0,0,0,1
4,1,0,1,0,6000,0,141,360,1,0,0,0,0,1


In [68]:

y.value_counts()


,count
Loan_Status,
1,422
0,192


model training -> Random Forest Classifier

In [69]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


In [74]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [75]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)

In [76]:
rf_model.fit(X_train, y_train) # train the model

RandomForestClassifier(random_state=42)

In [77]:
y_pred = rf_model.predict(X_test)

In [78]:
# validate model
from sklearn.metrics import accuracy_score
val_accuracy = accuracy_score(y_test, y_pred)

In [79]:
print("Validation Accuracy:", val_accuracy)

Validation Accuracy: 0.7723577235772358


In [80]:
# prediction on test data
test_predictions = rf_model.predict(test)

In [81]:
sample = pd.read_csv("/content/sample_submission.csv")

In [82]:
sample.head()

,Loan_ID,Loan_Status
0,LP001015,N
1,LP001022,N
2,LP001031,N
3,LP001035,N
4,LP001051,N


In [84]:
submission = pd.DataFrame({
    "Loan_ID": test_ids,  #  original IDs from  test data
    "Loan_Status": test_predictions     # model predictions
})

In [85]:
#converitng again back to y/n
submission["Loan_Status"] = submission["Loan_Status"].map({1: "Y", 0: "N"})


In [87]:
# Save CSV for submission
submission.to_csv("loan_predic_submsn.csv", index=False)
print("Submission file created successfully!")

Submission file created successfully!
